In [0]:
%run /Users/lshao10@jh.edu/ARF_NIV_Target_Trial_Emulation_DatabricksV/code/0_cohort_construction

In [0]:
# --- Load required dependenies ---
import yaml
import pathlib
import pandas as pd
from datetime import datetime
from time import perf_counter

# Count elapsed time
notebook_start = perf_counter()

# --- Load global and local configuration ---
# Project root
project_root = pathlib.Path.cwd()

# Checking configuration files
if not isinstance(config, dict):
    raise ValueError("config.yaml must contain key-value pairs")




# Load configuration files
while not (project_root / "config").is_dir():
  if project_root.parent == project_root:
    raise Exception("Configuration folder not found")
  project_root = project_root.parent

with open(project_root / "config" / "config.yaml", encoding = "utf-8") as f:
  config = yaml.safe_load(f)

with open(project_root / "config" / "global_config.yaml", encoding = "utf-8") as f:
  global_config = yaml.safe_load(f)

# Assign global config values
config_sql = {key: value for key, value in global_config.items() if key != "required_tables"}

# Assign local config values
tables_location = config["tables_location"]
project_location = config["project_location"]
institution = config["institution"]
time_zone = config["time_zone"]
file_type = config["file_type"]

# --- Loading project related clinical thresholds ---
while not (project_root / "outlier-thresholds").is_dir():
  if project_root.parent == project_root:
    raise Exception("Threshold folder not found")
  project_root = project_root.parent

thresholds_path = project_root / "outlier-thresholds" / "project_outlier_thresholds.csv"
thresholds = pd.read_csv(thresholds_path).set_index("variable_name")

# --- Expose configuration and/or thresholds to SQL as needed ---
config_sql["age_min"] = float(thresholds.loc["age_at_admission", "lower_limit"])
config_sql["age_max"] = float(thresholds.loc["age_at_admission", "upper_limit"])
config_sql["ph_cutoff"] = float(thresholds.loc["ph_cutoff", "lower_limit"])
config_sql["sf_cutoff"] = float(thresholds.loc["sf_cutoff", "lower_limit"])
config_sql["spo2_eligibility"] = float(thresholds.loc["spo2_eligibility", "upper_limit"])
config_sql["ph_arterial_min"] = float(thresholds.loc["ph_arterial", "lower_limit"])
config_sql["ph_arterial_max"] = float(thresholds.loc["ph_arterial", "upper_limit"])
config_sql["ph_venous_min"] = float(thresholds.loc["ph_venous", "lower_limit"])
config_sql["ph_venous_max"] = float(thresholds.loc["ph_venous", "upper_limit"])
config_sql["spo2_min"] = float(thresholds.loc["spo2", "lower_limit"])
config_sql["spo2_max"] = float(thresholds.loc["spo2", "upper_limit"])
config_sql["fio2_min"] = float(thresholds.loc["fio2_set", "lower_limit"])
config_sql["fio2_max"] = float(thresholds.loc["fio2_set", "upper_limit"])

spark.createDataFrame([config_sql]).createOrReplaceTempView("config_sql")

# --- Load CLIF tables ---
# Check if tables exist
clif_table_filenames = {item.name.removesuffix("/") for item in dbutils.fs.ls(tables_location) if item.isDir()}

required_tables = set(global_config["required_tables"])

missing_table = sorted(required_tables - clif_table_filenames)

if len(missing_table) > 0:
  raise Exception(f"Missing tables: {missing_table}")
else: 
  print("All tables present.")

# Register source CLIF tables as temp views
for table_name in required_tables:
  source = spark.read.parquet(f"{tables_location}/{table_name}")
  source.createOrReplaceTempView(f"src_{table_name}")
  print(f"Registered temp view for {table_name}")
